# 02 · Unexpected zeros and metacell size targets (Kolla et al. 2020, E16)

Notebook 01 showed that dropout is mostly explained by sequencing depth, and that no single metacell size suits every cell type. This notebook follows up on three things:

- **A. Fix the leverage reference.** Notebook 01 weighted genes by each cell type's own mean, so a gene absent from a cell type got zero weight there and stayed invisible. Here we also use the whole-dataset mean.
- **B. Explain the unexpected zeros.** Some highly expressed genes (e.g. *Ldha*, *Pkm*, *Cpt2* in hair cells) are zero in 15–22% of cells where depth predicts almost none. Are those zeros in the same cells? Do they follow cell quality, replicate, or a region of the t-SNE?
- **C. Set metacell size targets per cell type**, with a cap so every type keeps several metacells, and list the genes pooling can't rescue.

Outputs are written as CSVs to `OUT_DIR`.

## 0 · Settings

In [ ]:
import os, sys, json

BASE = '/scratch/prj/crb_inner_ear/k2147692/metabolic'
REPO_DIR = f'{BASE}/code/Metabolic-pipeline'
DATA_PATH = f'{BASE}/data/kolla/kolla_E16.h5ad'
OUT_DIR = f'{BASE}/results/02_unexpected_zeros_E16'

CELLTYPE_COL = 'cell_type'
SYMBOL_COL = 'gene_symbol'
REPLICATE_COL = 'replicate'
CLUSTER_COL = 'louvain'
TSNE_COLS = ('tSNE_1', 'tSNE_2')
SPECIES = 'mmusculus'
AND_STRATEGY = 'median'
OR_STRATEGY = 'sum'

FOCUS_GROUPS = ['IHC', 'OHC_1', 'OHC_2', 'Hensen', 'GER']   # hair cells, Hensen cells, and GER as a large contrast
LEVERAGE_MIN = 0.5        # genes that matter for reaction scores
EXCESS_MIN = 0.10         # at least 10 percentage points more zeros than depth predicts
WELL_EXPRESSED_MAX = 10   # genes detectable by pooling <= this many cells: zeros here are surprising
DETECT_PROB_MIN = 0.9     # a zero is 'unexpected' if depth predicted detection with >= 90% probability

COVERAGE = 0.8            # share of reachable leverage each metacell should detect
MIN_METACELLS = 3         # fewest metacells to keep per cell type
REFERENCE_SIZE = 50       # current SEACells target size, for comparison

sys.path.insert(0, REPO_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, chi2_contingency
from metabolic_tools.metacell_diagnostics import (
    recover_counts, cell_qc, gene_dropout_leverage, dropout_diagnostic,
    unexpected_zeros, metacell_size_targets)

adata = sc.read_h5ad(DATA_PATH)
report = recover_counts(adata)
print('Count recovery integer fraction:', report['integer_fraction'])

qc = cell_qc(adata, symbol_col=SYMBOL_COL)
adata.obs[qc.columns] = qc
adata.obs.groupby(CELLTYPE_COL, observed=True)[['total_counts', 'n_genes', 'pct_mito']].median().round(1)

## A · Leverage: cell-type reference vs whole-dataset reference
- **group**: rules evaluated on each cell type's mean (as in notebook 01). Genes absent from a type get zero weight there.
- **global**: rules evaluated on the whole-dataset mean, so a gene's importance comes from its role in the model and stays the same across cell types.

`leverage_never_detected` was always 0 in notebook 01. With the global reference it shows how much reaction-relevant expression is completely absent from each cell type; pooling can't fix that.

In [ ]:
lev_group, adata_model = gene_dropout_leverage(adata, CELLTYPE_COL, species=SPECIES, symbol_col=SYMBOL_COL,
                                               and_strategy=AND_STRATEGY, or_strategy=OR_STRATEGY, reference='group')
lev_global, _ = gene_dropout_leverage(adata, CELLTYPE_COL, species=SPECIES, symbol_col=SYMBOL_COL,
                                      and_strategy=AND_STRATEGY, or_strategy=OR_STRATEGY, reference='global')

genes_group, summary_group = dropout_diagnostic(adata_model, lev_group, CELLTYPE_COL, reference_size=REFERENCE_SIZE)
genes_global, summary_global = dropout_diagnostic(adata_model, lev_global, CELLTYPE_COL, reference_size=REFERENCE_SIZE)

compare = pd.DataFrame({
    'n_cells': summary_global['n_units'],
    'zero_frac (group)': summary_group['weighted_zero_frac'],
    'zero_frac (global)': summary_global['weighted_zero_frac'],
    'never_detected (global)': summary_global['leverage_never_detected'],
    f'covered_at_{REFERENCE_SIZE} (group)': summary_group[f'leverage_covered_at_{REFERENCE_SIZE}'],
    f'covered_at_{REFERENCE_SIZE} (global)': summary_global[f'leverage_covered_at_{REFERENCE_SIZE}'],
}).sort_values('never_detected (global)', ascending=False)
compare.round(3)

In [ ]:
# Important genes completely absent from a cell type (no counts in any cell of that type)
absent = genes_global[np.isinf(genes_global['units_needed']) & (genes_global['leverage'] >= 1)]
print(f'{absent["model_gene"].nunique()} genes with leverage >= 1 are absent from at least one cell type')
(absent.groupby('symbol')['group'].agg(lambda g: ', '.join(sorted(g)))
       .rename('absent from').to_frame()
       .assign(n_types=lambda d: d['absent from'].str.count(',') + 1)
       .sort_values('n_types', ascending=False).head(30))

## B · Unexpected zeros: whose cells are they?
For each focus cell type we use two gene sets:
- **flagged genes**: leverage ≥ 0.5 and ≥ 10 points more zeros than depth predicts (the genes notebook 01 highlighted);
- **well-expressed genes**: all reaction-relevant genes detectable by pooling ≤ 10 cells. Zeros in these are surprising in any cell.

For every cell we count **unexpected zeros** (zero despite ≥ 90% predicted detection) and compare with the count expected by chance. If the extra zeros pile up in particular cells, the next plots show what those cells have in common.

In [ ]:
flagged = genes_group[(genes_group['leverage'] >= LEVERAGE_MIN) & (genes_group['excess_zero_frac'] >= EXCESS_MIN)]
flagged = flagged.sort_values(['group', 'excess_zero_frac'], ascending=[True, False])
print(flagged.groupby('group').size().rename('flagged genes').to_string())
flagged[flagged['group'].isin(FOCUS_GROUPS)][['group', 'symbol', 'leverage', 'zero_frac', 'expected_zero_frac', 'excess_zero_frac', 'units_needed']].round(3)

In [ ]:
cell_scores = {}
for group in FOCUS_GROUPS:
    well = genes_group[(genes_group['group'] == group) & (genes_group['leverage'] > 0)
                       & (genes_group['units_needed'] <= WELL_EXPRESSED_MAX)]['model_gene'].tolist()
    unexp, prob = unexpected_zeros(adata_model, well, CELLTYPE_COL, group, min_detect_prob=DETECT_PROB_MIN)
    eligible = prob >= DETECT_PROB_MIN
    observed = unexp.sum(axis=1)
    expected = ((1 - prob) * eligible).sum(axis=1)

    df = adata_model.obs.loc[unexp.index, [REPLICATE_COL, CLUSTER_COL, *TSNE_COLS, 'total_counts', 'n_genes', 'pct_mito']].copy()
    df['n_eligible_genes'] = eligible.sum(axis=1)
    df['unexpected_zeros'] = observed
    df['expected_unexpected_zeros'] = expected
    df['excess'] = observed - expected
    df['high'] = observed > expected + 3 * np.sqrt(expected) + 1
    cell_scores[group] = df
    print(f'{group:8s} {len(well):3d} well-expressed genes | cells: {len(df):4d} | '
          f'mean unexpected zeros {observed.mean():.2f} vs expected {expected.mean():.2f} | '
          f'high-excess cells: {df["high"].sum()} ({df["high"].mean():.0%})')

### B1 · Do the flagged genes drop out in the same cells?
Correlation between genes' unexpected-zero patterns across cells. **Warm blocks** mean the same cells lack several genes together, pointing to a cell-level cause. **Near-zero** correlations mean the zeros are scattered, i.e. gene-level noise.

In [ ]:
panels = [g for g in FOCUS_GROUPS if g in set(flagged['group'])]
fig, axes = plt.subplots(1, len(panels), figsize=(4.6 * len(panels), 4.4), squeeze=False)
for ax, group in zip(axes.flat, panels):
    genes = flagged.loc[flagged['group'] == group, 'model_gene'].tolist()
    unexp, _ = unexpected_zeros(adata_model, genes, CELLTYPE_COL, group, min_detect_prob=0.0)
    unexp.columns = lev_group.loc[unexp.columns, 'symbol'].values
    corr = unexp.astype(float).corr().fillna(0)
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap='RdBu_r')
    ax.set_xticks(range(len(corr)), corr.columns, rotation=90, fontsize=7)
    ax.set_yticks(range(len(corr)), corr.index, fontsize=7)
    off_diag = corr.values[~np.eye(len(corr), dtype=bool)]
    ax.set_title(f'{group}: median r = {np.median(off_diag):.2f}' if len(off_diag) else group, fontsize=9)
fig.colorbar(im, ax=axes, shrink=0.7, label='correlation of zero patterns')
plt.show()

### B2 · Do cells with extra zeros look low-quality?
Damaged or empty-droplet-like cells tend to have **fewer UMIs, fewer genes and a higher mitochondrial fraction**. If `excess` tracks these, the zeros are technical and those cells should be filtered or down-weighted before pooling.

In [ ]:
rows = []
fig, axes = plt.subplots(len(FOCUS_GROUPS), 3, figsize=(11, 2.8 * len(FOCUS_GROUPS)), squeeze=False)
for r, group in enumerate(FOCUS_GROUPS):
    df = cell_scores[group]
    for c, col in enumerate(['total_counts', 'n_genes', 'pct_mito']):
        rho, p = spearmanr(df[col], df['excess'])
        rows.append({'group': group, 'qc_metric': col, 'spearman_rho': rho, 'p_value': p})
        ax = axes[r, c]
        ax.scatter(df[col], df['excess'], s=6, alpha=0.5, c=np.where(df['high'], 'tab:red', 'tab:grey'), linewidths=0)
        ax.set_title(f'{group} · rho = {rho:.2f}', fontsize=9)
        ax.set_xlabel(col, fontsize=8)
        if col == 'total_counts':
            ax.set_xscale('log')
        if c == 0:
            ax.set_ylabel('excess unexpected zeros', fontsize=8)
plt.tight_layout()
plt.show()
qc_assoc = pd.DataFrame(rows)
qc_assoc.pivot(index='group', columns='qc_metric', values='spearman_rho').round(2)

### B3 · Are they concentrated in one replicate?
A strong skew towards one replicate suggests a batch effect. If so, the metacell method should avoid pooling across replicates, or the data need batch handling first.

In [ ]:
rep_rows = []
for group in FOCUS_GROUPS:
    df = cell_scores[group]
    tab = pd.crosstab(df[REPLICATE_COL], df['high'])
    p = chi2_contingency(tab)[1] if tab.shape == (tab.shape[0], 2) and tab.shape[0] > 1 else np.nan
    by_rep = df.groupby(REPLICATE_COL, observed=True).agg(cells=('high', 'size'), high_share=('high', 'mean'), mean_excess=('excess', 'mean'))
    by_rep['group'] = group
    by_rep['chi2_p'] = p
    rep_rows.append(by_rep.reset_index())
replicate_table = pd.concat(rep_rows, ignore_index=True)[['group', REPLICATE_COL, 'cells', 'high_share', 'mean_excess', 'chi2_p']]
replicate_table.round(3)

### B4 · Do they sit together on the t-SNE, or in one Louvain cluster?
If high-excess cells form a patch while looking normal on QC, they're likely a **real subpopulation**, and metacells must not pool across that boundary.

In [ ]:
tsne_x, tsne_y = TSNE_COLS
fig, axes = plt.subplots(1, len(FOCUS_GROUPS), figsize=(3.6 * len(FOCUS_GROUPS), 3.6), squeeze=False)
for ax, group in zip(axes.flat, FOCUS_GROUPS):
    df = cell_scores[group]
    ax.scatter(adata_model.obs[tsne_x], adata_model.obs[tsne_y], s=1, c='lightgrey', linewidths=0)
    sca = ax.scatter(df[tsne_x], df[tsne_y], s=6, c=df['excess'], cmap='magma_r', linewidths=0)
    lo_x, hi_x = df[tsne_x].quantile([0.01, 0.99])
    lo_y, hi_y = df[tsne_y].quantile([0.01, 0.99])
    pad_x, pad_y = (hi_x - lo_x) * 0.3 + 1, (hi_y - lo_y) * 0.3 + 1
    ax.set_xlim(lo_x - pad_x, hi_x + pad_x)
    ax.set_ylim(lo_y - pad_y, hi_y + pad_y)
    ax.set_title(group, fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(sca, ax=ax, shrink=0.7)
plt.tight_layout()
plt.show()

cluster_table = pd.concat([
    cell_scores[g].groupby(CLUSTER_COL, observed=True).agg(cells=('high', 'size'), high_share=('high', 'mean'), mean_excess=('excess', 'mean'))
                  .query('cells >= 10').assign(group=g).reset_index()
    for g in FOCUS_GROUPS], ignore_index=True)[['group', CLUSTER_COL, 'cells', 'high_share', 'mean_excess']]
cluster_table.round(3)

### B · How to read these results
| What you see | Likely cause | What it means for metacells |
|---|---|---|
| Excess rises as `total_counts` / `n_genes` fall, or `pct_mito` rises | Low-quality or damaged cells | Filter or down-weight those cells before pooling |
| High-excess cells concentrated in one replicate | Batch effect | Don't pool across replicates (or correct batch first) |
| A patch on the t-SNE or one Louvain cluster, with normal QC | Real subpopulation | Metacells must stay within the subpopulation |
| Flagged genes correlated in B1, no QC/replicate/t-SNE pattern | Cell-to-cell variation in a shared programme | Pool nearby cells on a metabolic-aware graph |
| Flagged genes uncorrelated, no pattern anywhere | Gene-level overdispersion | Safe to pool; a negative-binomial model would fit better than Poisson |

## C · Metacell size targets per cell type
Uses the **global** leverage reference. For each cell type:
- **cap_cells** = n_cells / `MIN_METACELLS`, so every type keeps at least that many metacells.
- **target_cells** = smallest pool detecting `COVERAGE` of the *reachable* leverage with 95% probability.
- **target_umis** = the same target as a UMI budget. An adaptive method should aim for this, since deeper cells need fewer partners.
- **unreachable** genes need more cells than the cap. They shouldn't drive metacell size; flag their reactions for wider flux bounds instead.

In [ ]:
targets, uncertain = metacell_size_targets(genes_global, summary_global, coverage=COVERAGE, min_metacells=MIN_METACELLS)
targets[f'covered_at_{REFERENCE_SIZE}'] = summary_global[f'leverage_covered_at_{REFERENCE_SIZE}']
targets.sort_values('target_cells', ascending=False).round(3)

In [ ]:
# How sensitive are the targets to the coverage goal and the minimum number of metacells?
grid = []
for cov in (0.7, 0.8, 0.9):
    for k in (2, 3, 5):
        t, _ = metacell_size_targets(genes_global, summary_global, coverage=cov, min_metacells=k)
        grid.append(t[['target_cells', 'target_umis', 'leverage_unreachable']].assign(coverage=cov, min_metacells=k).reset_index())
sensitivity = pd.concat(grid, ignore_index=True)
sensitivity.pivot_table(index='group', columns=['coverage', 'min_metacells'], values='target_cells')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
order = targets.sort_values('median_library_size').index
ax.scatter(targets.loc[order, 'median_library_size'], targets.loc[order, 'target_cells'], s=np.sqrt(targets.loc[order, 'n_cells']) * 3, alpha=0.7)
for g in order:
    ax.annotate(g, (targets.loc[g, 'median_library_size'], targets.loc[g, 'target_cells']), fontsize=7, xytext=(4, 3), textcoords='offset points')
ax.axhline(REFERENCE_SIZE, color='grey', ls='--', lw=0.8)
ax.set_xlabel('median UMIs per cell')
ax.set_ylabel(f'target metacell size (cells, {COVERAGE:.0%} coverage)')
ax.set_title('Deeper cell types need smaller metacells (point size = number of cells)', fontsize=10)
plt.show()

In [ ]:
# Genes pooling can't rescue, and the reactions they feed
uncertain_genes = (uncertain.groupby(['model_gene', 'symbol'])
                   .agg(n_cell_types=('group', 'nunique'), cell_types=('group', lambda g: ', '.join(sorted(g))),
                        leverage=('leverage', 'first'))
                   .reset_index()
                   .join(lev_global['reactions'], on='model_gene')
                   .sort_values(['n_cell_types', 'leverage'], ascending=False))
print(f'{len(uncertain_genes)} genes are unreachable in at least one cell type')
uncertain_genes.head(30)

## Save

In [ ]:
compare.to_csv(os.path.join(OUT_DIR, 'A_leverage_reference_comparison.csv'))
absent.to_csv(os.path.join(OUT_DIR, 'A_absent_genes.csv'), index=False)
lev_global.to_csv(os.path.join(OUT_DIR, 'gene_leverage_global.csv'))
genes_global.to_csv(os.path.join(OUT_DIR, 'gene_dropout_global.csv'), index=False)
summary_global.to_csv(os.path.join(OUT_DIR, 'dropout_summary_global.csv'))
flagged.to_csv(os.path.join(OUT_DIR, 'B_flagged_genes.csv'), index=False)
qc_assoc.to_csv(os.path.join(OUT_DIR, 'B_qc_association.csv'), index=False)
replicate_table.to_csv(os.path.join(OUT_DIR, 'B_replicate_table.csv'), index=False)
cluster_table.to_csv(os.path.join(OUT_DIR, 'B_cluster_table.csv'), index=False)
pd.concat(cell_scores, names=['group', 'cell']).to_csv(
    os.path.join(OUT_DIR, 'B_cell_scores.csv'))
targets.to_csv(os.path.join(OUT_DIR, 'C_size_targets.csv'))
sensitivity.to_csv(os.path.join(OUT_DIR, 'C_size_sensitivity.csv'), index=False)
uncertain_genes.to_csv(os.path.join(OUT_DIR, 'C_unreachable_genes.csv'), index=False)
print('Saved to', OUT_DIR)

## Sending results back
1. **File → Save Notebook As…** → `metabolic/results/02_unexpected_zeros_E16/02_unexpected_zeros_E16_run.ipynb`
2. In a terminal, zip the folder and reset the original notebook so the next `git pull` is clean:
```bash
cd /scratch/prj/crb_inner_ear/k2147692/metabolic/results && zip -r 02_unexpected_zeros_E16.zip 02_unexpected_zeros_E16
cd /scratch/prj/crb_inner_ear/k2147692/metabolic/code/Metabolic-pipeline && git checkout -- notebooks/
```
3. Right-click the zip in Jupyter's file browser → **Download**, then extract it into `Documents\Metabolic-results`.